# Proportion of Depth-Tuned Neurons vs Cortical Depth

This notebook plots the proportion of depth-tuned neurons across cortical depth bins.
It pools data from:
1. `depth_mismatch_seq` (the 24 SfN poster sessions, using cell `z_coor` for depth)
2. `ccyp_l5_3d_vision` (using session `nominal_depth` from flexilims)
3. `colasa_3d-vision_revisions` (using session `nominal_depth` from flexilims, filtered such that selected sessions for each mouse are at least 50µm apart)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import flexiznam as flz
from cottage_analysis.io_module import suite2p as s2p_io

import matplotlib.font_manager as fm

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'


In [ ]:
# Project lists
PROJECTS = ["depth_mismatch_seq", "ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]

# Hardcoded original depth_mismatch_seq sessions
original_sfn_sessions = [
    "BRAC9972.2a_S20241017", "BRAC9972.2a_S20241021", "BRAC9972.2a_S20241026",
    "BRAC9972.2a_S20241028", "BRAC9972.2a_S20241030", "BRAC9972.2a_S20241101", 
    "BRAC9972.2c_S20241102", "BRAC9972.2c_S20241108", "BRAC9972.2c_S20241116",
    "BRAC9972.2c_S20241118", "BRAC9972.2c_S20241122", "BRAC9972.2c_S20241123", 
    "BRAC10658.3h_S20250418", "BRAC10658.3h_S20250421", "BRAC10658.3h_S20250424",
    "BRAC10658.3h_S20250428", "BRAC10658.3f_S20250501", "BRAC10658.3f_S20250513",
    "BRAC10658.3f_S20250517", "BRAC10658.3f_S20250529", "BRAC10514.4d_S20250501",
    "BRAC10514.4d_S20250513", "BRAC10514.4d_S20250517", "BRAC10514.4d_S20250528",
]

# Dictionary to store sessions and their nominal depths grouped by project
session_dict = {}

# 1. Add original depth_mismatch_seq sessions
flm_depth = flz.get_flexilims_session(project_id="depth_mismatch_seq")
all_depth_sessions = flz.get_entities(datatype="session", flexilims_session=flm_depth)
session_dict["depth_mismatch_seq"] = []
for sess in original_sfn_sessions:
    nd = all_depth_sessions.loc[sess, "nominal_depth"] if "nominal_depth" in all_depth_sessions.columns else np.nan
    if isinstance(nd, (list, np.ndarray, pd.Series)):
        nd = np.mean(nd)
    session_dict["depth_mismatch_seq"].append((sess, nd))

# 2. Query and filter ccyp_l5_3d_vision and colasa_3d-vision_revisions sessions
for proj in ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]:
    flm_sess = flz.get_flexilims_session(project_id=proj)
    sessions = flz.get_entities(datatype="session", flexilims_session=flm_sess)
    recordings = flz.get_entities(datatype="recording", flexilims_session=flm_sess)
    
    candidates = []
    for session_name, sess_row in sessions.iterrows():
        nominal_depth = sess_row.get("nominal_depth", None)
        if isinstance(nominal_depth, (list, np.ndarray, pd.Series)):
            if len(nominal_depth) == 0 or pd.isna(nominal_depth).all():
                continue
            mean_depth = np.mean(nominal_depth)
        else:
            if pd.isna(nominal_depth) or nominal_depth is None:
                continue
            mean_depth = nominal_depth
            
        sess_id = sess_row["id"]
        sess_recs = recordings[recordings["origin_id"] == sess_id]
        if sess_recs.empty:
            continue
            
        protocols = sess_recs["protocol"].dropna().unique() if "protocol" in sess_recs.columns else []
        names = sess_recs["name"].dropna().unique()
        
        has_spheres = "SpheresPermTubeReward" in protocols or any("SpheresPermTubeReward" in n for n in names)
        has_multidepth = any("multidepth" in p.lower() for p in protocols) or any("multidepth" in n.lower() for n in names)
        has_openloop = any("playback" in p.lower() for p in protocols) or any("playback" in n.lower() for n in names)
        is_treadmill_only = all("SpheresTubeMotor" in p or "Treadmill" in p for p in protocols) if len(protocols) > 0 else False
        
        if has_spheres and not has_multidepth and not has_openloop and not is_treadmill_only:
            # check neurons_df dataset existence
            neurons_ds = flz.get_datasets(
                origin_name=session_name,
                dataset_type="neurons_df",
                flexilims_session=flm_sess,
                allow_multiple=True,
            )
            if len(neurons_ds) > 0:
                candidates.append((session_name, mean_depth))
                
    if proj == "colasa_3d-vision_revisions":
        # Apply 50um apart filter per mouse
        df_candidates = pd.DataFrame(candidates, columns=["session", "nominal_depth"])
        df_candidates["mouse"] = df_candidates["session"].apply(lambda x: x.split("_")[0])
        selected_candidates = []
        for mouse, group in df_candidates.groupby("mouse"):
            group_sorted = group.sort_values("nominal_depth")
            mouse_selected = []
            for idx, row in group_sorted.iterrows():
                if not mouse_selected:
                    mouse_selected.append(row)
                else:
                    if all(abs(row.nominal_depth - sel.nominal_depth) >= 50.0 for sel in mouse_selected):
                        mouse_selected.append(row)
            selected_candidates.extend(mouse_selected)
        session_dict[proj] = [(row.session, row.nominal_depth) for row in selected_candidates]
    else:
        session_dict[proj] = candidates

print("Sessions compiled successfully.")

In [ ]:
# Load all neurons_df datasets
all_neurons_list = []

for proj in PROJECTS:
    flm_sess = flz.get_flexilims_session(project_id=proj)
    sess_list = session_dict[proj]
    
    for sess, nominal_depth in tqdm(sess_list, desc=f"Loading {proj}"):
        neurons_ds = flz.get_datasets(
            origin_name=sess,
            dataset_type="neurons_df",
            flexilims_session=flm_sess,
            allow_multiple=True,
        )
        tmp = pd.read_pickle(neurons_ds[0].path_full)
        
        # Load iscell if missing
        if "iscell" not in tmp.columns:
            suite2p_ds = flz.get_datasets(
                origin_name=sess,
                dataset_type="suite2p_rois",
                flexilims_session=flm_sess,
                exclude_datasets={"annotated": True},
                allow_multiple=True,
            )
            if not suite2p_ds:
                suite2p_ds = flz.get_datasets(
                    origin_name=sess,
                    dataset_type="suite2p_rois",
                    flexilims_session=flm_sess,
                    allow_multiple=True,
                )
            iscell_array = s2p_io.load_is_cell(suite2p_ds[0].path_full)
            if iscell_array.ndim == 2:
                tmp["iscell"] = tmp["roi"].apply(lambda r: iscell_array[int(r), 0] if not pd.isna(r) else False)
            else:
                tmp["iscell"] = tmp["roi"].apply(lambda r: iscell_array[int(r)] if not pd.isna(r) else False)
            
        tmp["session"] = sess
        tmp["project"] = proj
        tmp["nominal_depth"] = nominal_depth
        tmp["roi_uid"] = sess + "_" + tmp.roi.astype(str)
        tmp["mouse"] = sess.split("_")[0]
        all_neurons_list.append(tmp)

neurons_df = pd.concat(all_neurons_list, ignore_index=True)
print(f"\n{len(neurons_df)} total neurons loaded across {neurons_df.session.nunique()} sessions")

In [ ]:
# Configuration: Grouping level for plotting (can be set to 'session' or 'mouse')
GROUP_BY = 'session'
neurons_df["is_depth_neuron"] = (
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop"] < 0.05)
)
# Filter for cells only (iscell == 1/True)
subset = neurons_df[neurons_df["iscell"].astype(int) == 1].copy()

# Define the depth column: use cell z_coor for depth_mismatch_seq, nominal_depth for others
subset["depth"] = np.where(
    subset["project"] == "depth_mismatch_seq",
    subset["z_coor"],
    subset["nominal_depth"]
)

print(f'Number of mice: {len(subset.mouse.unique())}')
# Drop cells without valid coordinates or depth
subset = subset.dropna(axis=0, subset=["depth"], how="any").copy()
print(f'Number of mice: {len(subset.mouse.unique())}')

# Bin neurons by cortical depth
zcoor_bins = np.arange(0, 550, 50)
subset.loc[:, "binned_zcoor"] = pd.cut(subset["depth"], bins=zcoor_bins)

# Identify depth-tuned neurons (Spearman correlation criterion)
subset.loc[:, "is_depth_tuned"] = subset.apply(
    lambda x: True if ((x["depth_tuning_test_spearmanr_rval_closedloop"] > 0.1) & (x["depth_tuning_test_spearmanr_pval_closedloop"] < 0.05)) else False, 
    axis=1
)

# Compute the proportion of depth-tuned neurons
summary = subset.groupby([GROUP_BY, 'binned_zcoor'])[[GROUP_BY, "binned_zcoor", "is_depth_tuned"]].apply(
    lambda x: x['is_depth_tuned'].sum() / len(x) if len(x) > 0 else np.nan
).to_frame("proportion_depth_tuned").reset_index()

# Filter out empty bins
print(f'Number of mice: {len(subset.mouse.unique())}')
summary = summary.dropna(subset=["proportion_depth_tuned"])
print(f'Number of mice: {len(subset.mouse.unique())}')
# Calculate bin centers for plotting
summary.loc[:, "bin_centre"] = summary["binned_zcoor"].apply(lambda x: x.mid)

# Reformat props and bin centers for boxplot/scatter plotting
props = []
bin_centres = []
for g in sorted(summary.groupby("bin_centre").groups.keys()):
    props.append(summary.groupby("bin_centre").get_group(g)["proportion_depth_tuned"].values)
    bin_centres.append(summary.groupby("bin_centre").get_group(g)["bin_centre"].values[0])
print(f'Number of mice: {len(subset.mouse.unique())}')

In [ ]:
# Create the figure
cm = 1/2.54  # centimeters in inches
fig, pop_tuned = plt.subplots(figsize=(15*cm, 12*cm))

bin_vector = np.concatenate([np.repeat(b, len(props[i])) for i, b in enumerate(bin_centres)])
props_concat = np.concatenate(props)

# Plot median lines
pop_tuned.scatter(
    y = [np.nanmedian(p) for p in props],
    x = bin_centres,
    color = '''k''',
    marker='''_''',
    s=200,
    linewidth=3,
    zorder=3,
)

# Plot jittered individual sessions/mice
jitter_strength = 3
jittered_x = bin_vector + np.random.uniform(-jitter_strength, jitter_strength, size=bin_vector.shape)
pop_tuned.scatter(
    y = props_concat,
    x = jittered_x,
    color='''lightgray''',
    ec='''k''',
    s=32,
    alpha=0.8,
    zorder=2,
)

# Format axes
pop_tuned.set_xticks(bin_centres)
pop_tuned.set_xticklabels([f"{int(b-25)}-{int(b+25)}" for b in bin_centres], rotation=45, fontsize=10)
pop_tuned.tick_params(axis='''y''', which='''major''', labelsize=10)
pop_tuned.set_ylabel("Proportion of depth-tuned neurons", fontsize=12)
pop_tuned.set_xlabel("Cortical depth bin (µm)", fontsize=12)
pop_tuned.set_ylim([0, 1.05])
sns.despine(ax=pop_tuned)

# Add sample sizes (total number of cells in each bin)
subset.loc[:, "bin_centre"] = subset["binned_zcoor"].apply(lambda x: x.mid)
for ibin, bin_val in enumerate(bin_centres):
    total_neurons = subset[(subset['''bin_centre'''] == bin_val)].shape[0]
    n_group = len(props[ibin])
    # Add text label above points
    pop_tuned.text(
        x=bin_val,
        y=np.max(props[ibin]) + 0.05 if len(props[ibin]) > 0 else 0.1,
        s=f"N={total_neurons}\n({n_group})",
        ha='''center''',
        fontsize=7,
        zorder=4,
    )

plt.tight_layout()

# Save and show figure
try:
    save_dir = Path("/nemo/lab/znamenskiyp/home/shared/projects/v1_depth_seq/poster_figures")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(save_dir / "depth_tuned_proportion_pooled.svg", dpi=300, bbox_inches='''tight''')
    plt.savefig(save_dir / "depth_tuned_proportion_pooled.pdf", dpi=300, bbox_inches='''tight''')
    print(f"Saved figure to {save_dir}")
except Exception as e:
    print(f"Could not save to poster_figures directory, saving locally. Error: {e}")
    plt.savefig("depth_tuned_proportion_pooled.svg", dpi=300, bbox_inches='''tight''')

plt.show()

## Option A: Plotting Mean connected by a thick line (without sample size numbers)

In [ ]:
# Create the figure
cm = 1/2.54  # centimeters in inches
fig, pop_tuned = plt.subplots(figsize=(15*cm, 12*cm))

bin_vector = np.concatenate([np.repeat(b, len(props[i])) for i, b in enumerate(bin_centres)])
props_concat = np.concatenate(props)

# Plot jittered individual sessions/mice
jitter_strength = 3
jittered_x = bin_vector + np.random.uniform(-jitter_strength, jitter_strength, size=bin_vector.shape)
pop_tuned.scatter(
    y = props_concat,
    x = jittered_x,
    color='''lightgray''',
    ec='''k''',
    s=32,
    alpha=0.8,
    zorder=2,
)

# Calculate and plot mean values as a thick connected line
bin_means = [np.nanmean(p) for p in props]
pop_tuned.plot(
    bin_centres,
    bin_means,
    color='''k''',
    linewidth=3,
    zorder=3,
)
pop_tuned.scatter(
    bin_centres,
    bin_means,
    color='''k''',
    s=60,
    zorder=4,
)
layer_colors = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}

pop_tuned.axvspan(150, 345, color=layer_colors['Layer 2/3'], alpha=0.3, zorder=1)
pop_tuned.axvspan(355, 510, color=layer_colors['Layer 5'], alpha=0.3, zorder=1)
# Format axes
pop_tuned.set_xticks(bin_centres)
pop_tuned.set_xticklabels([f"{int(b-25)}-{int(b+25)}" for b in bin_centres], rotation=45, fontsize=10)
pop_tuned.tick_params(axis='''y''', which='''major''', labelsize=10)
pop_tuned.set_ylabel("Proportion of depth-tuned neurons", fontsize=12)
pop_tuned.set_xlabel("Cortical depth bin (µm)", fontsize=12)
pop_tuned.set_ylim([0, 1.05])
sns.despine(ax=pop_tuned)

plt.tight_layout()

# Save and show figure
try:
    save_dir = Path("/nemo/lab/znamenskiyp/home/shared/projects/v1_depth_seq/poster_figures")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(save_dir / "depth_tuned_proportion_pooled_mean_line.svg", dpi=300, bbox_inches='''tight''')
    plt.savefig(save_dir / "depth_tuned_proportion_pooled_mean_line.pdf", dpi=300, bbox_inches='''tight''')
    print(f"Saved figure to {save_dir}")
except Exception as e:
    print(f"Could not save to poster_figures directory, saving locally. Error: {e}")
    plt.savefig("depth_tuned_proportion_pooled_mean_line.svg", dpi=300, bbox_inches='''tight''')

plt.show()

## Option B: Plotting Mean as wide horizontal bars / hlines (without sample size numbers)

In [ ]:
# Create the figure
cm = 1/2.54  # centimeters in inches
fig, pop_tuned = plt.subplots(figsize=(15*cm, 12*cm))

bin_vector = np.concatenate([np.repeat(b, len(props[i])) for i, b in enumerate(bin_centres)])
props_concat = np.concatenate(props)

# Plot jittered individual sessions/mice
jitter_strength = 3
jittered_x = bin_vector + np.random.uniform(-jitter_strength, jitter_strength, size=bin_vector.shape)
pop_tuned.scatter(
    y = props_concat,
    x = jittered_x,
    color='''lightgray''',
    ec='''k''',
    s=32,
    alpha=0.8,
    zorder=2,
)

# Plot mean values as wide horizontal bars
bin_means = [np.nanmean(p) for p in props]
pop_tuned.hlines(
    y=bin_means,
    xmin=np.array(bin_centres) - 15,
    xmax=np.array(bin_centres) + 15,
    colors='''k''',
    linewidth=4,
    zorder=3,
)

# Format axes
pop_tuned.set_xticks(bin_centres)
pop_tuned.set_xticklabels([f"{int(b-25)}-{int(b+25)}" for b in bin_centres], rotation=45, fontsize=10)
pop_tuned.tick_params(axis='''y''', which='''major''', labelsize=10)
pop_tuned.set_ylabel("Proportion of depth-tuned neurons", fontsize=12)
pop_tuned.set_xlabel("Cortical depth bin (µm)", fontsize=12)
pop_tuned.set_ylim([0, 1.05])
sns.despine(ax=pop_tuned)

plt.tight_layout()

# Save and show figure
try:
    save_dir = Path("/nemo/lab/znamenskiyp/home/shared/projects/v1_depth_seq/poster_figures")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(save_dir / "depth_tuned_proportion_pooled_mean_hlines.svg", dpi=300, bbox_inches='''tight''')
    plt.savefig(save_dir / "depth_tuned_proportion_pooled_mean_hlines.pdf", dpi=300, bbox_inches='''tight''')
    print(f"Saved figure to {save_dir}")
except Exception as e:
    print(f"Could not save to poster_figures directory, saving locally. Error: {e}")
    plt.savefig("depth_tuned_proportion_pooled_mean_hlines.svg", dpi=300, bbox_inches='''tight''')

plt.show()

# Distribution of preferred depth

Splitting layer 2/3 (150-350um) and layer 5 (>350um)

In [ ]:
# Split neurons into layer 2/3 (cortical depth 150-350 um) and layer 5 (>350 um),
# then look at the distribution of their preferred (virtual) depth.

# Preferred virtual depth of the closed-loop tuning fit (in metres -> convert to cm).
PREF_COL = "preferred_depth_closedloop"

# Only depth-tuned neurons have a meaningful preferred depth.
dist = subset[subset["is_depth_tuned"] & subset[PREF_COL].notna()].copy()


def assign_layer(cortical_depth):
    if 150 <= cortical_depth <= 350:
        return "Layer 2/3"
    elif cortical_depth > 350:
        return "Layer 5"
    return np.nan


dist["layer"] = dist["depth"].apply(assign_layer)
dist = dist.dropna(subset=["layer"]).copy()

# Preferred depth in cm on a log axis.
dist["log_pref_depth"] = np.log10(dist[PREF_COL] * 100)

print("Depth-tuned neurons per layer:")
print(dist["layer"].value_counts())


In [ ]:
# Preferred virtual depth vs cortical depth, coloured by layer
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(12 * cm, 10 * cm))
ax.scatter(dist.log_pref_depth, dist.depth, c=dist.layer.map(layer_colors),
           alpha=0.1, s=10, edgecolor='none')

# x-axis: preferred depth in cm on a log10 scale
tick_pos = [10, 100, 1000]
ax.set_xticks(np.log10(tick_pos))
ax.set_xticklabels(tick_pos)
minor_ticks = np.log10(
    np.concatenate((np.arange(2, 10) * 1, np.arange(2, 10) * 10, np.arange(2, 10) * 100, [2000]))
)
ax.set_xticks(minor_ticks, minor=True)

ax.set_xlabel('Preferred virtual depth (cm)')
ax.set_ylabel('Cortical depth (µm)')
ax.invert_yaxis()  # deeper (larger) cortical depth at the bottom
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Density version: 2D histogram of preferred virtual depth vs cortical depth
fig, ax = plt.subplots(figsize=(12 * cm, 10 * cm))

d = dist.dropna(subset=["log_pref_depth", "depth"])
hb = ax.hexbin(d.log_pref_depth, d.depth, gridsize=40, cmap="magma", mincnt=1, bins="log")
cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("Neuron count (log)")

# x-axis: preferred depth in cm on a log10 scale
tick_pos = [10, 100, 1000]
ax.set_xticks(np.log10(tick_pos))
ax.set_xticklabels(tick_pos)
minor_ticks = np.log10(
    np.concatenate((np.arange(2, 10) * 1, np.arange(2, 10) * 10, np.arange(2, 10) * 100, [2000]))
)
ax.set_xticks(minor_ticks, minor=True)

ax.set_xlabel('Preferred virtual depth (cm)')
ax.set_ylabel('Cortical depth (µm)')
ax.invert_yaxis()  # deeper (larger) cortical depth at the bottom
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Density normalised within each cortical-depth slice:
# for every cortical depth bin, show the conditional distribution of preferred virtual
# depth (each row sums to 1), removing the effect of unequal neuron counts across depth.
fig, ax = plt.subplots(figsize=(12 * cm, 10 * cm))

d = dist.dropna(subset=["log_pref_depth", "depth"])

x_edges = np.linspace(np.log10(2), np.log10(2000), 31)      # preferred depth (log10 cm)
y_edges = np.arange(0, d["depth"].max() + 50, 50)           # cortical depth slices (µm)

H, _, _ = np.histogram2d(d.log_pref_depth, d.depth, bins=[x_edges, y_edges])
# Normalise each cortical-depth slice (column over x) to sum to 1
col_sums = H.sum(axis=0, keepdims=True)
H_norm = np.divide(H, col_sums, out=np.full_like(H, np.nan), where=col_sums > 0)

pcm = ax.pcolormesh(x_edges, y_edges, H_norm.T, cmap="magma", shading="flat")
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label("Proportion of neurons\nwithin cortical slice")

# x-axis: preferred depth in cm on a log10 scale
tick_pos = [10, 100, 1000]
ax.set_xticks(np.log10(tick_pos))
ax.set_xticklabels(tick_pos)
minor_ticks = np.log10(
    np.concatenate((np.arange(2, 10) * 1, np.arange(2, 10) * 10, np.arange(2, 10) * 100, [2000]))
)
ax.set_xticks(minor_ticks, minor=True)

ax.set_xlabel('Preferred virtual depth (cm)')
ax.set_ylabel('Cortical depth (µm)')
ax.invert_yaxis()  # deeper (larger) cortical depth at the bottom
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Plot preferred-depth distributions for L2/3 vs L5 (one axis per layer, side by side)
from scipy.stats import mannwhitneyu

layer_colors = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}

cm = 1 / 2.54
fig, axes = plt.subplots(1, 2, figsize=(30 * cm, 8 * cm), sharey=True)

nbins = 20
tol = 1e-4
# The depth-tuning fit clips preferred depth to [0.02, 20] m = [2, 2000] cm
# (find_depth_neurons.fit_preferred_depth defaults). Anchor the axis to those fixed
# clip bounds -- not the data min/max -- so the 10/100/1000 cm ticks always land at the
# same positions, and railed cells pile up exactly at the N.P./F.P. edges.
min_d = np.log10(2)      # 2 cm    -> near rail (N.P.)
max_d = np.log10(2000)   # 2000 cm -> far rail (F.P.)
interior_bins = np.linspace(min_d, max_d, nbins)
bar_w = (max_d - min_d) / nbins
gap = 2.5 * bar_w                 # visual separation between main hist and extreme bars
np_x = min_d - gap               # N.P. bar centre (near preference)
fp_x = max_d + gap               # F.P. bar centre (far preference)

tick_pos = [10, 100, 1000]
minor_ticks = np.log10(
    np.concatenate(
        (np.arange(2, 10) * 1, np.arange(2, 10) * 10, np.arange(2, 10) * 100, [2000])
    )
)

for ax, layer in zip(axes, ["Layer 2/3", "Layer 5"]):
    ld = dist[dist["layer"] == layer]
    vals = ld["log_pref_depth"].values
    n = len(vals)                # normalise by ALL cells in the layer -> proportions sum to 1
    near = vals <= min_d + tol   # railed to nearest depth
    far = vals >= max_d - tol    # railed to farthest depth
    interior = vals[~(near | far)]
    color = layer_colors[layer]
    ax.hist(
        interior,
        bins=interior_bins,
        weights=np.ones(interior.size) / n,
        color=color,
        edgecolor=color,
        alpha=0.45,
    )
    # Extreme bins (clipped cells)
    ax.bar(np_x, near.sum() / n, width=bar_w, color=color, edgecolor=color, alpha=0.45, zorder=2)
    ax.bar(fp_x, far.sum() / n, width=bar_w, color=color, edgecolor=color, alpha=0.45, zorder=2)
    ax.plot(np.median(vals), 1.0, marker="v", color=color, markersize=12,
            transform=ax.get_xaxis_transform(), clip_on=False, zorder=4)

    # Visual divider between the interior histogram and the extreme bars
    for xdiv in (min_d - gap / 2, max_d + gap / 2):
        ax.axvline(xdiv, color="lightgray", linewidth=1, zorder=1)

    ax.set_yticks([0, 0.04, 0.08])
    ax.tick_params(axis="y", labelsize=18)
    if ax is axes[0]:
        ax.set_ylabel("Proportion", fontsize=18)
    ax.set_title(layer, fontsize=18, color=color, loc="left")

    # x ticks: log-scaled depth (cm) in the interior, N.P./F.P. at the extremes
    ax.set_xticks([np_x] + list(np.log10(tick_pos)) + [fp_x])
    ax.set_xticklabels(["N.P."] + tick_pos + ["F.P."], fontsize=18)
    ax.set_xticks(minor_ticks, minor=True)
    ax.set_xlim(np_x - bar_w, fp_x + bar_w)
    ax.set_xlabel("Preferred virtual depth (cm)", fontsize=18)
    sns.despine(ax=ax)

plt.tight_layout()

# Compare distributions (full data incl. railed cells)
for layer in ["Layer 2/3", "Layer 5"]:
    ld = dist[dist["layer"] == layer]
    print(f"{layer}: median preferred depth {10 ** ld['log_pref_depth'].median():.1f} cm "
          f"(n={len(ld)} cells, {ld['session'].nunique()} sessions, {ld['mouse'].nunique()} mice)")
l23 = dist.loc[dist["layer"] == "Layer 2/3", "log_pref_depth"]
l5 = dist.loc[dist["layer"] == "Layer 5", "log_pref_depth"]
u_stat, pval = mannwhitneyu(l23, l5)
print(f"Mann-Whitney U: U={u_stat:.0f}, p={pval:.3g}")

# Save
try:
    save_dir = Path("/nemo/lab/znamenskiyp/home/shared/projects/v1_depth_seq/poster_figures")
    os.makedirs(save_dir, exist_ok=True)
    fig.savefig(save_dir / "preferred_depth_distribution_by_layer.svg", dpi=300, bbox_inches="tight")
    fig.savefig(save_dir / "preferred_depth_distribution_by_layer.pdf", dpi=300, bbox_inches="tight")
    print(f"Saved figure to {save_dir}")
except Exception as e:
    print(f"Could not save to poster_figures directory, saving locally. Error: {e}")
    plt.savefig("preferred_depth_distribution_by_layer.svg", dpi=300, bbox_inches="tight")

plt.show()
